# Sampling — Practice Problems with Solutions

**CSE 219: Signals and Linear Systems — Lab**

15 problems on sampling, in roughly increasing order of difficulty. Each problem comes in four cells:

1. the **statement**,
2. a **scaffold** cell for you to fill in,
3. the **solution** code,
4. a **discussion** of what the numbers mean.

Try the scaffold before scrolling into the solution. Only `numpy` and `matplotlib` are needed — every filter, transform and quantiser here is built from its defining formula so that nothing is a black box.

---

### The one idea behind the whole notebook

A computer cannot hold a continuous-time signal. So we fake $x(t)$ with a very fine grid (20 kHz), call that "analog", and study what happens when we pick a much coarser grid out of it:

$$x[n] = x(nT_s), \qquad T_s = \frac{1}{f_s}$$

The sampling theorem says that if $x(t)$ contains no frequency above $f_{\max}$, then $x(t)$ can be recovered exactly from $x[n]$ provided

$$f_s > 2 f_{\max}$$

Almost everything below is either a demonstration of this, a demonstration of what breaks when it fails, or a demonstration of a case where the naive reading of it is too pessimistic.

## Setup: shared helpers

Run this cell first. These are deliberately small and readable — read them once and you will know exactly what every later cell is doing.

| helper | what it does |
|---|---|
| `analog_grid(duration)` | fine grid standing in for continuous time |
| `sample_grid(duration, fs)` | the sampling instants $t_n = n/f_s$ |
| `cosine(f, t, phase, amp)` | $A\cos(2\pi f t + \phi)$ |
| `dft_magnitude(x, fs)` | magnitude spectrum with a real frequency axis in Hz |
| `plot_samples(...)` | "analog" curve with the samples stemmed on top |
| `stem_spectrum(...)` | one- or two-sided spectrum on an axis |
| `interior_rms(err, t)` | RMS over the middle of a record, ignoring edge effects |

Two notes on `dft_magnitude`. The DFT bin index $k$ maps to the physical frequency $f_k = k f_s / N$, which is what `np.fft.fftfreq` computes. And the factor $2/N$ is chosen so that a cosine of amplitude $A$ shows a peak of height $A$: the $1/N$ undoes the DFT's unnormalised sum, and the $2$ folds the negative-frequency twin back onto its positive partner.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# plot defaults shared by every figure in the notebook
plt.rcParams.update({
    "figure.figsize": (9, 3.6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 10,
    "font.size": 9,
})

FS_REF = 20000          # the "analog" reference rate: a grid this fine stands in for continuous time


def analog_grid(duration, fs_ref=FS_REF):
    """Fine time grid that we pretend is continuous time: t = 0, 1/fs_ref, 2/fs_ref, ..."""
    n_points = int(round(duration * fs_ref))
    return np.arange(n_points) / fs_ref


def sample_grid(duration, fs):
    """Sampling instants t_n = n / fs for n = 0 .. N-1, covering 0 <= t < duration."""
    n_samples = int(round(duration * fs))
    return np.arange(n_samples) / fs


def cosine(f, t, phase=0.0, amp=1.0):
    """A * cos(2*pi*f*t + phase), evaluated at every time in t."""
    return amp * np.cos(2 * np.pi * f * t + phase)


def dft_magnitude(x, fs, one_sided=True):
    """
    Magnitude spectrum on a physical frequency axis (Hz).

    Bin k sits at f_k = k * fs / N.  The result is scaled so that a cosine
    of amplitude A produces a peak of height A.
    """
    N = len(x)
    X = np.fft.fft(x) / N                  # 1/N undoes the DFT's unnormalised sum
    f = np.fft.fftfreq(N, d=1 / fs)        # bin index -> Hz

    order = np.argsort(f)                  # reorder to -fs/2 ... +fs/2
    f, X = f[order], X[order]

    mag = 2 * np.abs(X)                    # x2 folds the negative-frequency twin onto its partner
    if one_sided:
        keep = f >= 0
        return f[keep], mag[keep]
    return f, mag


def plot_samples(t, x, tn, xn, ax=None, title="", sig_label="x(t)"):
    """Analog curve in grey with the samples stemmed on top."""
    if ax is None:
        _, ax = plt.subplots()
    ax.plot(t, x, color="0.65", lw=1, label=sig_label)
    ax.stem(tn, xn, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="samples")
    ax.set_xlabel("t (s)")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper right")
    return ax


def stem_spectrum(x, fs, ax=None, one_sided=True, title=""):
    """Stem plot of the magnitude spectrum; returns (ax, f, mag) so callers can inspect peaks."""
    if ax is None:
        _, ax = plt.subplots()
    f, mag = dft_magnitude(x, fs, one_sided=one_sided)
    ax.stem(f, mag, basefmt=" ", markerfmt="C0.")
    ax.set_xlabel("f (Hz)")
    ax.set_ylabel("|X(f)|")
    ax.set_title(title)
    return ax, f, mag


def interior_rms(err, t, frac=0.6):
    """RMS of err over the middle `frac` of the record, ignoring the edges."""
    span = t[-1] - t[0]
    margin = (1 - frac) / 2 * span         # how much to skip at each end
    inside = (t >= t[0] + margin) & (t <= t[-1] - margin)
    return np.sqrt(np.mean(err[inside] ** 2))


print("helpers loaded")

---
## Problem 1 — Sampling a sinusoid: the picture

Let

$$x(t) = \cos(2\pi \cdot 5 t), \qquad 0 \le t < 1 \text{ s}$$

**(a)** Plot $x(t)$ on the fine analog grid.

**(b)** On the same axes, stem-plot the samples taken at $f_s = 50$ Hz, and in a second panel the samples at $f_s = 8$ Hz.

**(c)** For each rate report the sampling period $T_s$, the number of samples $N$, and the samples per period of the sinusoid,

$$\frac{\text{samples}}{\text{period}} = \frac{f_s}{f_0}$$

**Think:** how many samples per period do you need before the dots start to look like the cosine?

In [ ]:
# Your turn
t_dense = analog_grid(1.0)
x_dense = cosine(5, t_dense)

# for fs in (50, 8):
#     tn = sample_grid(1.0, fs)
#     xn = cosine(5, tn)
#     plot_samples(t_dense, x_dense, tn, xn, title=...)

### Solution 1

In [ ]:
# the "analog" signal: a 5 Hz cosine on the fine grid
f0 = 5
duration = 1.0
t_dense = analog_grid(duration)
x_dense = cosine(f0, t_dense)

# one panel per sampling rate
rates = (50, 8)
fig, axes = plt.subplots(len(rates), 1, figsize=(9, 6), sharex=True)

for ax, fs in zip(axes, rates):
    # take the samples x[n] = x(n / fs)
    tn = sample_grid(duration, fs)
    xn = cosine(f0, tn)

    # the numbers asked for in part (c)
    Ts = 1 / fs
    N = len(tn)
    samples_per_period = fs / f0

    plot_samples(t_dense, x_dense, tn, xn, ax=ax,
                 title=f"fs = {fs} Hz  |  Ts = {Ts:.4f} s  |  N = {N}  "
                       f"|  samples per period = {samples_per_period:.1f}")

fig.tight_layout()
plt.show()

**Discussion 1**

| $f_s$ | $T_s$ | $N$ | samples/period |
|---|---|---|---|
| 50 Hz | 0.02 s | 50 | 10 |
| 8 Hz | 0.125 s | 8 | 1.6 |

At 50 Hz the dots trace the cosine unambiguously. At 8 Hz there are fewer than two samples per period, and the dots suggest a much slower oscillation — a 3 Hz one, as Problem 3 will make precise.

Rule of thumb: more than 2 samples per period is *survival* (the information is there), about 10 is *comfortable viewing* (the raw dots look like the signal).

---
## Problem 2 — Finding the Nyquist boundary by experiment

Same 5 Hz cosine. Sample at $f_s = 20, 10, 9, 6$ Hz. For each rate, plot the samples joined by straight lines on top of the true $x(t)$.

**(a)** At which rates can you still recognise a 5 Hz oscillation?

**(b)** $f_s = 10$ Hz is exactly $2f_{\max}$. Repeat it with

$$x(t) = \cos(2\pi \cdot 5 t + \phi), \qquad \phi = 0,\ \tfrac{\pi}{4},\ \tfrac{\pi}{2}$$

What happens at $\phi = \pi/2$, and why does the theorem demand $f_s > 2f_{\max}$ strictly rather than $f_s \ge 2f_{\max}$?

In [ ]:
# Your turn
# for fs in (20, 10, 9, 6):
#     tn = sample_grid(1.0, fs)
#     ...

### Solution 2

In [ ]:
f0 = 5
rates = (20, 10, 9, 6)

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True, sharey=True)

for ax, fs in zip(axes.ravel(), rates):
    tn = sample_grid(1.0, fs)
    xn = cosine(f0, tn)

    # folding formula (Problem 3 turns this into a function)
    wrapped = f0 % fs                        # wrap into [0, fs)
    apparent = min(wrapped, fs - wrapped)    # reflect the top half down

    ax.plot(t_dense, x_dense, color="0.7", lw=1)    # true signal in grey
    ax.plot(tn, xn, "o-", ms=4, lw=1.2)             # samples joined by straight lines
    ax.set_title(f"fs = {fs} Hz   (apparent frequency {apparent} Hz)")

fig.suptitle("5 Hz cosine, samples joined by straight lines", fontsize=11)
fig.tight_layout()
plt.show()

# (b) the boundary case fs = 2*fmax: what you get depends on the phase
fs = 2 * f0
tn = sample_grid(1.0, fs)
phases = [(0.0, "0"), (np.pi / 4, "pi/4"), (np.pi / 2, "pi/2")]

print(f"fs = 2*fmax = {fs} Hz — effect of the phase we do not control:\n")
for phase, name in phases:
    xn = cosine(f0, tn, phase=phase)
    first_six = np.round(xn[:6], 3)
    peak = np.abs(xn).max()
    print(f"  phase = {name:>4}:  first samples = {first_six}   peak |x[n]| = {peak:.3f}")

**Discussion 2**

**(a)** Only $f_s = 20$ Hz is faithful. At $f_s = 9$ Hz the samples oscillate at 4 Hz and at $f_s = 6$ Hz at 1 Hz — visible, but wrong.

**(b)** At exactly two samples per period, we land on the same two points of the waveform forever:

$$x[n] = \cos(\pi n + \phi) = (-1)^n \cos\phi$$

so the recovered amplitude is $|\cos\phi|$ — full at $\phi = 0$, $0.707$ at $\phi = \pi/4$, and **exactly zero** at $\phi = \pi/2$, where every sample lands on a zero crossing and the signal vanishes completely.

Since $\phi$ is not something we control, $f_s = 2f_{\max}$ cannot be guaranteed to work. That is the whole reason the inequality is strict.

---
## Problem 3 — The folding formula

When a cosine of frequency $f_0$ is sampled at $f_s$, its samples are identical to those of some cosine whose frequency lies in $[0, f_s/2]$. Call that the **apparent** (aliased) frequency.

**(a)** Write `apparent_freq(f0, fs)` in two steps only:

$$f \leftarrow f_0 \bmod f_s \quad (\text{wrap into } [0, f_s)), \qquad
f_a = \begin{cases} f, & f \le f_s/2 \\ f_s - f, & f > f_s/2 \end{cases}$$

**(b)** Check that `apparent_freq(100, 150)` gives 50.

**(c)** Prove it numerically: with $f_s = 150$ Hz, verify that

$$\cos(2\pi \cdot 100 \, t_n) = \cos(2\pi \cdot 50 \, t_n), \qquad t_n = n/150$$

to machine precision, even though the two continuous signals are completely different.

**(d)** Tabulate `apparent_freq(f0, 100)` for $f_0 = 10, 40, 60, 90, 110, 140, 190$ Hz and plot apparent vs true frequency. What shape do you get?

This function is used by almost every later problem, so run the solution cell even if you skip the problem.

In [ ]:
def apparent_freq(f0, fs):
    """Frequency in [0, fs/2] whose samples match those of f0."""
    # TODO: two lines
    raise NotImplementedError

### Solution 3

In [ ]:
def apparent_freq(f0, fs):
    """Frequency in [0, fs/2] whose samples match those of f0."""
    f = np.mod(f0, fs)                        # step 1: wrap into [0, fs)
    return np.where(f > fs / 2, fs - f, f)    # step 2: fold the top half down


# (b) a single check
print("apparent_freq(100, 150) =", apparent_freq(100, 150))

# (c) numerical proof: two different cosines, identical samples
fs = 150
tn = np.arange(200) / fs
samples_100 = cosine(100, tn)
samples_50 = cosine(50, tn)
max_diff = np.abs(samples_100 - samples_50).max()
print(f"max |cos(2pi*100*t_n) - cos(2pi*50*t_n)| = {max_diff:.2e}\n")

# (d) a table of true vs apparent frequency ...
fs = 100
f0s = np.array([10, 40, 60, 90, 110, 140, 190])
f_apparent = apparent_freq(f0s, fs)

print(f"fs = {fs} Hz")
for f0, fa in zip(f0s, f_apparent):
    print(f"   f0 = {f0:4d} Hz  ->  apparent {fa:5.1f} Hz")

# ... and the folding diagram
f_scan = np.linspace(0, 250, 1001)
plt.figure()
plt.plot(f_scan, apparent_freq(f_scan, fs))
plt.axvline(fs / 2, color="C3", ls="--", lw=1)
plt.text(fs / 2 + 2, 5, "fs/2", color="C3")
plt.xlabel("true $f_0$ (Hz)")
plt.ylabel("apparent $f$ (Hz)")
plt.title(f"Folding at fs = {fs} Hz")
plt.tight_layout()
plt.show()

**Discussion 3**

The graph is a **triangle wave**: it rises with slope $+1$ up to $f_s/2$, falls back to $0$ at $f_s$, and repeats with period $f_s$.

Every frequency above $f_s/2$ is reflected back down onto one below it, which is why $f_s/2$ is called the **folding** or **Nyquist frequency**. Part (c) is the key fact: after sampling, the data itself carries no trace of which of the two frequencies was present. No algorithm can undo this — aliasing is irreversible, which is why it has to be prevented *before* sampling (Problem 11).

---
## Problem 4 — Two signals, one set of samples

Take $f_s = 10$ Hz and the pair $f_a = 7$ Hz, $f_b = 3$ Hz.

**(a)** Plot both continuous cosines on $0 \le t < 1$ and overlay the samples of each. The two sample sets must coincide exactly.

**(b)** Show on paper, then confirm in code, that

$$\cos\!\left(\frac{2\pi \cdot 7 n}{10}\right) = \cos\!\left(\frac{2\pi \cdot 3 n}{10}\right)$$

*Hint:* $\tfrac{7}{10} = 1 - \tfrac{3}{10}$, and $\cos(2\pi n - \theta) = \cos\theta$ for integer $n$.

**(c)** Repeat with sines. Are the sample sets still equal, or equal up to a sign? What does that sign mean physically?

In [ ]:
# Your turn

### Solution 4

In [ ]:
fs = 10
f_high, f_low = 7, 3        # f_low = fs - f_high, so the two alias onto each other
tn = sample_grid(1.0, fs)

plt.figure(figsize=(9, 4))
# the two continuous signals
plt.plot(t_dense, cosine(f_high, t_dense), color="C3", lw=1, label=f"cos(2π·{f_high}t)")
plt.plot(t_dense, cosine(f_low, t_dense), color="C0", lw=1, label=f"cos(2π·{f_low}t)")
# their samples: the white crosses land exactly inside the black circles
plt.plot(tn, cosine(f_high, tn), "ko", ms=8, label=f"samples of {f_high} Hz")
plt.plot(tn, cosine(f_low, tn), "wx", ms=6, mew=2, label=f"samples of {f_low} Hz")
plt.xlabel("t (s)")
plt.title(f"fs = {fs} Hz: two different signals, identical samples")
plt.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

# cosines: the samples are identical
cos_diff = np.abs(cosine(f_high, tn) - cosine(f_low, tn)).max()
print(f"max |cos7[n] - cos3[n]| = {cos_diff:.2e}")

# sines: the samples are negatives of each other
sin_high = np.sin(2 * np.pi * f_high * tn)
sin_low = np.sin(2 * np.pi * f_low * tn)
print(f"max |sin7[n] - sin3[n]| = {np.abs(sin_high - sin_low).max():.3f}")
print(f"max |sin7[n] + sin3[n]| = {np.abs(sin_high + sin_low).max():.2e}")

**Discussion 4**

**(b)** At the sampling instants $t_n = n/10$,

$$2\pi \cdot 7 \cdot \frac{n}{10} = 2\pi n - 2\pi \cdot 3 \cdot \frac{n}{10}$$

and since $\cos(2\pi n - \theta) = \cos\theta$ for integer $n$, the two sample sequences are identical.

**(c)** For sines, $\sin(2\pi n - \theta) = -\sin\theta$, so the 7 Hz samples are **minus** the 3 Hz samples — same magnitude, flipped sign.

That sign is the physically interesting part. The alias of the 7 Hz tone is really sitting at $-3$ Hz, not $+3$ Hz. A cosine is an even function and cannot distinguish $+f$ from $-f$; a sine can. Problem 13 turns this sign into something you can literally watch: the direction a wheel appears to spin.

---
## Problem 5 — Aliasing of a frequency sweep

A linear chirp sweeps from 0 to 400 Hz over $T = 2$ s, so its instantaneous frequency is

$$f_{\text{inst}}(t) = k t, \qquad k = \frac{400}{2} = 200 \text{ Hz/s}$$

The phase is the integral of the instantaneous frequency, so

$$\theta(t) = 2\pi \int_0^t f_{\text{inst}}(\tau)\,d\tau = 2\pi \frac{k}{2} t^2,
\qquad x(t) = \cos\!\left(2\pi \frac{k}{2} t^2\right)$$

Derive that phase yourself before coding it. Sample at $f_s = 500$ Hz.

**(a)** Plot $f_{\text{inst}}(t)$ — a straight line from 0 to 400.

**(b)** Using `apparent_freq`, plot the apparent frequency against $t$.

**(c)** Where does the track turn around, and why exactly there?

In [ ]:
# Your turn
T, k = 2.0, 200.0
# tn = sample_grid(T, 500)
# f_true = k * tn
# f_app  = apparent_freq(f_true, 500)

### Solution 5

In [ ]:
duration = 2.0
k = 200.0        # chirp rate: instantaneous frequency f_inst(t) = k * t   (Hz)
fs = 500

tn = sample_grid(duration, fs)
f_true = k * tn                          # where the frequency really is
f_app = apparent_freq(f_true, fs)        # where the samples say it is

t_turnaround = (fs / 2) / k              # the moment f_inst reaches fs/2

plt.figure(figsize=(9, 4))
plt.plot(tn, f_true, "--", label="true $f_{inst}(t) = 200t$")
plt.plot(tn, f_app, lw=2, label=f"apparent frequency, fs = {fs} Hz")
plt.axhline(fs / 2, color="0.5", lw=1)
plt.text(0.03, fs / 2 + 8, f"fs/2 = {fs // 2} Hz", color="0.4")
plt.axvline(t_turnaround, color="C3", ls=":", lw=1)
plt.xlabel("t (s)")
plt.ylabel("Hz")
plt.title("A chirp folds back at the Nyquist frequency")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"turn-around at t = (fs/2)/k = {fs / 2:.0f}/{k:.0f} = {t_turnaround} s")

**Discussion 5**

The true frequency crosses $f_s/2 = 250$ Hz at

$$t = \frac{250}{200} = 1.25 \text{ s}$$

which is exactly where the apparent track reverses. Played as audio, the sweep rises for 1.25 s and then *descends* — the classic aliasing fold. If the sweep continued past 500 Hz it would bounce off 0 Hz and rise again, reproducing the triangle of Problem 3(d).

To hear it, write both versions to WAV and compare:

```python
from scipy.io import wavfile
t_hi = analog_grid(T)
wavfile.write("chirp_clean.wav", FS_REF, (0.5*np.cos(2*np.pi*(k/2)*t_hi**2)).astype(np.float32))
wavfile.write("chirp_alias.wav", fs,     (0.5*np.cos(2*np.pi*(k/2)*tn**2)).astype(np.float32))
```

---
## Problem 6 — Seeing aliasing in the spectrum

$x(t) = \cos(2\pi \cdot 300 t)$, observed for 0.5 s.

**(a)** Sample at $f_s = 1000$ Hz and plot the one-sided magnitude spectrum. Where is the peak?

**(b)** Sample at $f_s = 400$ Hz. Predict the peak with `apparent_freq` *before* running, then confirm.

**(c)** Plot the two-sided spectrum for the $f_s = 400$ case. Which peak is "the" signal?

**(d)** Change the record length to 0.503 s. The peak smears out. This is **leakage**, not aliasing — state the difference in one sentence.

In [ ]:
# Your turn
# for fs in (1000, 400):
#     tn = sample_grid(0.5, fs)
#     stem_spectrum(cosine(300, tn), fs, title=...)

### Solution 6

In [ ]:
f0 = 300
duration = 0.5
rates = (1000, 400)

# (a), (b): one-sided spectrum at each rate
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, fs in zip(axes, rates):
    tn = sample_grid(duration, fs)
    xn = cosine(f0, tn)
    _, f, mag = stem_spectrum(xn, fs, ax=ax)

    measured_peak = f[np.argmax(mag)]                 # frequency of the tallest stem
    predicted_peak = float(apparent_freq(f0, fs))     # from the folding formula
    ax.set_title(f"fs = {fs} Hz — peak at {measured_peak:.1f} Hz "
                 f"(predicted {predicted_peak:.1f} Hz)")

fig.suptitle(f"{f0} Hz cosine, one-sided spectrum", fontsize=11)
fig.tight_layout()
plt.show()

# (c) two-sided view of the aliased case
fs = 400
tn = sample_grid(duration, fs)
alias = float(apparent_freq(f0, fs))
stem_spectrum(cosine(f0, tn), fs, one_sided=False,
              title=f"fs = {fs} Hz, two-sided: peaks at ±{alias:.0f} Hz")
plt.tight_layout()
plt.show()

# (d) a record that does not hold a whole number of periods
fs = 1000
tn = sample_grid(0.503, fs)
f, mag = dft_magnitude(cosine(f0, tn), fs)

plt.figure()
plt.plot(f, mag)
plt.xlim(f0 - 50, f0 + 50)
plt.xlabel("f (Hz)")
plt.title(f"0.503 s record: leakage skirts around {f0} Hz")
plt.tight_layout()
plt.show()

**Discussion 6**

**(a)** $f_s = 1000 > 2 \cdot 300$, so the peak sits at the true 300 Hz.

**(b)** $f_s = 400 < 600$, so 300 Hz folds to $400 - 300 = 100$ Hz. Once sampled, the data is honestly indistinguishable from a real 100 Hz tone.

**(c)** The peaks at $\pm 100$ Hz are a conjugate pair; neither is more real than the other. A real-valued signal always has $X(-f) = X^*(f)$, so the physical tone *is* the pair.

**(d)** Aliasing is a **sampling** error: distinct frequencies become identical before the DFT ever runs, and it is irreversible. Leakage is a **windowing** artefact: the frequency is still correct, but a 0.503 s record contains a non-integer number of periods so no DFT bin lands exactly on 300 Hz and the energy spreads into neighbours. Leakage is fixed by a longer record or a taper window.

---
## Problem 7 — Spectral replication: why images appear at multiples of $f_s$

Sampling makes the spectrum periodic with period $f_s$:

$$X_s(f) = \frac{1}{T_s}\sum_{m=-\infty}^{\infty} X(f - m f_s)$$

Let

$$x(t) = \cos(2\pi \cdot 30 t) + 0.6\cos(2\pi \cdot 80 t) + 0.3\cos(2\pi \cdot 140 t)
\qquad (B = 140 \text{ Hz})$$

**(a)** Sample at $f_s = 400$ Hz ($> 2B$). Plot the magnitude on an extended axis from $-800$ to $800$ Hz by **tiling** the computed two-sided spectrum — since it repeats every $f_s$, just concatenate shifted copies. Mark the baseband copy.

**(b)** Repeat with $f_s = 200$ Hz ($< 2B$). Which components from neighbouring copies land inside baseband? Predict each with `apparent_freq`.

**(c)** State the condition on $f_s$ in terms of $B$ for the copies never to touch.

In [ ]:
# Your turn
def tiled_spectrum(xn, fs, n_copies=2):
    """Repeat the two-sided spectrum every fs to expose the images."""
    # TODO
    raise NotImplementedError

### Solution 7

In [ ]:
def tiled_spectrum(xn, fs, n_copies=2):
    """Repeat the two-sided spectrum every fs to expose the images."""
    f, mag = dft_magnitude(xn, fs, one_sided=False)

    # shift the frequency axis by m*fs for m = -n_copies .. +n_copies
    shifts = range(-n_copies, n_copies + 1)
    f_tiled = np.concatenate([f + m * fs for m in shifts])
    mag_tiled = np.concatenate([mag for m in shifts])    # same magnitudes, repeated

    order = np.argsort(f_tiled)
    return f_tiled[order], mag_tiled[order]


def x_multi(t):
    """Three tones; the highest is 140 Hz, so the bandwidth B = 140 Hz."""
    return cosine(30, t) + 0.6 * cosine(80, t) + 0.3 * cosine(140, t)


B = 140
rates = (400, 200)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for ax, fs in zip(axes, rates):
    tn = sample_grid(1.0, fs)
    f_tiled, mag_tiled = tiled_spectrum(x_multi(tn), fs)

    ax.plot(f_tiled, mag_tiled, lw=0.9)
    ax.axvspan(-fs / 2, fs / 2, color="C2", alpha=0.12)      # baseband, shaded
    for m in (-2, -1, 0, 1, 2):
        ax.axvline(m * fs, color="0.7", lw=0.8, ls=":")      # multiples of fs
    ax.set_xlim(-800, 800)
    ax.set_xlabel("f (Hz)")
    ax.set_title(f"fs = {fs} Hz   (B = {B} Hz, 2B = {2 * B} Hz)   — shaded = baseband")

fig.tight_layout()
plt.show()

fs = 200
print(f"Landing spots at fs = {fs} Hz:")
for f0 in (30, 80, 140):
    print(f"   {f0:3d} Hz  ->  {float(apparent_freq(f0, fs)):5.1f} Hz")

**Discussion 7**

**(a)** At $f_s = 400$ the copies centred at $0, \pm 400, \pm 800$ are cleanly separated. Baseband occupies $[-140, 140]$ and the next copy starts at $400 - 140 = 260$ Hz, leaving a 120 Hz guard band on each side.

**(b)** At $f_s = 200$ the copy centred at 200 Hz reaches down to $200 - 140 = 60$ Hz and spills into baseband. The 140 Hz tone therefore appears at 60 Hz, while 30 and 80 Hz (both below $f_s/2 = 100$) stay put. The 60 Hz intruder is now permanently inseparable from a genuine 60 Hz component.

**(c)** The copies never touch when the right edge of baseband clears the left edge of the next copy:

$$f_s - B > B \quad \Longleftrightarrow \quad f_s > 2B$$

which is the sampling theorem seen entirely in the frequency domain.

---
## Problem 8 — Ideal reconstruction by sinc interpolation

The reconstruction formula of the sampling theorem:

$$x_r(t) = \sum_{n} x[n] \, \operatorname{sinc}\!\left(\frac{t - nT_s}{T_s}\right),
\qquad \operatorname{sinc}(u) = \frac{\sin(\pi u)}{\pi u}$$

`np.sinc` already uses this normalised definition.

**(a)** Implement `sinc_reconstruct(samples, fs, t)` in one vectorised line, building an $N_{\text{samples}} \times N_t$ matrix of sinc values and contracting it against the samples.

**(b)** For $x(t) = \cos(2\pi \cdot 5 t)$ on $[0,1)$, sample at $f_s = 20$ Hz, reconstruct on a dense grid, and plot $x$, $x_r$ and the error.

**(c)** The error is worst near $t = 0$ and $t = 1$. That is **truncation**, not aliasing. Recompute the RMS using only the middle 60% (`interior_rms`).

**(d)** Repeat at $f_s = 8$ Hz. The interior error no longer vanishes. What is the leftover error oscillating at?

Later problems reuse this function, so run the solution cell if you skip ahead.

In [ ]:
def sinc_reconstruct(samples, fs, t):
    """x_r(t) = sum_n x[n] * sinc((t - n/fs) * fs)"""
    # TODO: one line using broadcasting
    raise NotImplementedError

### Solution 8

In [ ]:
def sinc_reconstruct(samples, fs, t):
    """x_r(t) = sum_n x[n] * sinc((t - n/fs) * fs)"""
    n = np.arange(len(samples))
    sample_times = n / fs

    # kernel[i, j] = sinc((t[j] - sample_times[i]) * fs), shape (N_samples, N_t)
    kernel = np.sinc((t[None, :] - sample_times[:, None]) * fs)

    # the matrix product does the sum over n for every t at once
    return samples @ kernel


f0 = 5
duration = 1.0
rates = (20, 8)

# a coarser reference grid: the kernel is N_samples x N_t, so keep N_t modest
t_r = analog_grid(duration, fs_ref=4000)
x_true = cosine(f0, t_r)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for ax, fs in zip(axes, rates):
    tn = sample_grid(duration, fs)
    xn = cosine(f0, tn)

    x_rec = sinc_reconstruct(xn, fs, t_r)
    err = x_true - x_rec
    full_rms = np.sqrt(np.mean(err ** 2))
    inner_rms = interior_rms(err, t_r)

    ax.plot(t_r, x_true, color="0.65", lw=1.5, label="x(t)")
    ax.plot(t_r, x_rec, "C0", lw=1.2, label="sinc reconstruction")
    ax.plot(t_r, err, "C3", lw=1, label="error")
    ax.plot(tn, xn, "k.", ms=6)
    ax.set_title(f"fs = {fs} Hz — full RMS = {full_rms:.4f}, interior RMS = {inner_rms:.4f}")
    ax.legend(fontsize=8, loc="upper right")

axes[-1].set_xlabel("t (s)")
fig.tight_layout()
plt.show()

print("apparent frequency at fs = 8 Hz:", float(apparent_freq(f0, 8)), "Hz")

**Discussion 8**

**$f_s = 20$ Hz:** the interior error is around $10^{-2}$ and shrinks further if the record is lengthened. What remains is truncation — the exact formula needs samples from $-\infty$ to $+\infty$, and $\operatorname{sinc}$ decays only as $1/t$, so the missing tails still matter near the edges.

**$f_s = 8$ Hz:** the interior error stays of order 1. The reconstruction is a clean 3 Hz cosine, since $\text{apparent\_freq}(5, 8) = 3$, so the error oscillates at the beat between 5 Hz and 3 Hz. No amount of extra data fixes it: the information was destroyed at the instant of sampling.

This is the practical difference between the two error sources — truncation shrinks with a longer record, aliasing does not shrink with anything.

---
## Problem 9 — Reconstruction error vs sampling rate

$$x(t) = \cos(2\pi \cdot 3 t) + 0.5\cos(2\pi \cdot 10 t - 0.4) \qquad \Rightarrow \qquad f_{\max} = 10 \text{ Hz}$$

Sweep $f_s$ from 5 to 100 Hz in 1 Hz steps. For each rate: sample, sinc-reconstruct on a dense grid, and take the interior RMS error.

**(a)** Plot the error against $f_s$ with a logarithmic $y$-axis.

**(b)** Locate the knee. Is it exactly at $f_s = 20$ Hz?

**(c)** Above the knee the error does not fall to zero. Why not?

In [ ]:
# Your turn

### Solution 9

In [ ]:
def x9(t):
    """Two tones; fmax = 10 Hz, so the Nyquist rate is 20 Hz."""
    return cosine(3, t) + 0.5 * cosine(10, t, phase=-0.4)


duration = 1.0
t_r = analog_grid(duration, fs_ref=2000)
x_true9 = x9(t_r)

# sample, reconstruct and measure the error at every integer rate 5 .. 100 Hz
fs_list = np.arange(5, 101)
errs = np.empty(len(fs_list))
for i, fs in enumerate(fs_list):
    xn = x9(sample_grid(duration, fs))
    x_rec = sinc_reconstruct(xn, fs, t_r)
    errs[i] = interior_rms(x_true9 - x_rec, t_r)

plt.figure(figsize=(9, 4))
plt.semilogy(fs_list, errs, ".-")
plt.axvline(20, color="C3", ls="--", label="$2f_{max}$ = 20 Hz")
plt.xlabel("$f_s$ (Hz)")
plt.ylabel("interior RMS error")
plt.title("Reconstruction error collapses once $f_s$ passes the Nyquist rate")
plt.legend()
plt.tight_layout()
plt.show()

for fs in (15, 19, 20, 21, 25, 40, 60):
    err_at_fs = errs[fs_list == fs][0]        # boolean mask picks out one entry
    print(f"   fs = {fs:3d} Hz  ->  error {err_at_fs:.4e}")

**Discussion 9**

**(b)** Not exactly at 20. Below 20 Hz the error is of order $0.5$: the 10 Hz component is simply aliased away. At $f_s = 20 = 2f_{\max}$ the error is still around $0.13$ — this is the boundary case of Problem 2(b), where two samples per period leave the recovered amplitude at the mercy of a phase we do not control. The error only settles onto its floor by about $f_s = 25$ Hz. The strict inequality $f_s > 2f_{\max}$ is visible as that offset.

**(c)** Above the knee the floor (around $10^{-2}$) is set by truncation, exactly as in Problem 8(c) — we sum finitely many sinc terms instead of infinitely many. It barely depends on $f_s$, which is why the curve flattens instead of continuing downward. Lengthen the record and the whole floor drops.

---
## Problem 10 — Three ways to join the dots

Reconstruct the same samples three ways:

1. **Zero-order hold (ZOH)** — hold each sample until the next, $x_{\text{zoh}}(t) = x\!\left[\lfloor t/T_s \rfloor\right]$. This is what a real DAC does.
2. **Linear interpolation** — `np.interp`.
3. **Ideal sinc interpolation** — Problem 8.

Use $x(t) = \cos(2\pi \cdot 5 t)$, $f_s = 30$ Hz, on $[0,1)$.

**(a)** Plot all three against the true $x(t)$.

**(b)** Report the interior RMS error of each and rank them.

**(c)** A ZOH is a convolution with a rectangle of width $T_s$, whose transform is $T_s\operatorname{sinc}(f T_s)$. So the in-band amplitude should droop by $|\operatorname{sinc}(f T_s)|$. Measure the 5 Hz peak of the ZOH output against the true one and check it against the formula.

In [ ]:
# Your turn
# idx   = np.clip((t_r / Ts).astype(int), 0, len(xn) - 1)
# x_zoh = xn[idx]

### Solution 10

In [ ]:
fs, f0 = 30, 5
Ts = 1 / fs
duration = 1.0

t_r = analog_grid(duration, fs_ref=6000)
x_true10 = cosine(f0, t_r)
tn = sample_grid(duration, fs)
xn = cosine(f0, tn)

# 1. zero-order hold: at each fine time, hold the most recent sample
most_recent = (t_r / Ts).astype(int)                  # index of the last sample before t
most_recent = np.clip(most_recent, 0, len(xn) - 1)    # stay inside the array
x_zoh = xn[most_recent]

# 2. linear: straight lines between neighbouring samples
x_lin = np.interp(t_r, tn, xn)

# 3. ideal: sinc interpolation from Problem 8
x_sinc = sinc_reconstruct(xn, fs, t_r)

plt.figure(figsize=(9, 4.5))
plt.plot(t_r, x_true10, color="0.65", lw=2.5, label="x(t)")
plt.plot(t_r, x_zoh, lw=1, label="zero-order hold")
plt.plot(t_r, x_lin, lw=1, label="linear")
plt.plot(t_r, x_sinc, lw=1, label="sinc")
plt.xlim(0, 0.4)
plt.xlabel("t (s)")
plt.title(f"Three reconstructions, fs = {fs} Hz")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("interior RMS error")
for name, x_rec in [("ZOH", x_zoh), ("linear", x_lin), ("sinc", x_sinc)]:
    print(f"   {name:8s}: {interior_rms(x_true10 - x_rec, t_r):.4f}")

# ZOH droop: compare the height of the f0 peak with and without the hold
f_zoh, mag_zoh = dft_magnitude(x_zoh, 6000)
f_true, mag_true = dft_magnitude(x_true10, 6000)
bin_zoh = np.argmin(np.abs(f_zoh - f0))         # bin nearest f0
bin_true = np.argmin(np.abs(f_true - f0))
measured_droop = mag_zoh[bin_zoh] / mag_true[bin_true]
predicted_droop = abs(np.sinc(f0 * Ts))         # ZOH frequency response is sinc(f*Ts)

print(f"\nmeasured droop at {f0} Hz = {measured_droop:.4f}")
print(f"|sinc(f0*Ts)|          = {predicted_droop:.4f}")

**Discussion 10**

Ranking: **sinc < linear < ZOH**. Sinc is exact in the interior; linear is a cheap and decent approximation; the staircase is the crudest.

The droop check comes out at

$$|\operatorname{sinc}(f_0 T_s)| = \left|\operatorname{sinc}\!\left(\tfrac{5}{30}\right)\right|
= \frac{\sin(\pi/6)}{\pi/6} = 0.9549$$

matching the measurement. This is the **aperture droop** of a real DAC, and it is why converters are followed by an inverse-sinc compensation filter. The ZOH also leaves images near multiples of $f_s$ (the staircase edges), which is why a smoothing reconstruction filter follows the DAC as well.

---
## Problem 11 — Why the anti-aliasing filter comes *before* the sampler

$$x(t) = \cos(2\pi \cdot 10 t) + 0.8\cos(2\pi \cdot 90 t)$$

recorded at $f_{s1} = 1000$ Hz for 1 s. We want to reduce the rate to $f_{s2} = 100$ Hz, i.e. decimate by $M = 10$.

Note the trap: at 100 Hz the 90 Hz tone aliases to exactly $100 - 90 = 10$ Hz, landing right on top of the tone we want.

**(a)** Naive route: keep every 10th sample, `x[::10]`. Plot the spectrum. What is the amplitude at 10 Hz, and why is it wrong?

**(b)** Build a lowpass FIR by hand (windowed sinc). The ideal lowpass impulse response with cutoff $f_c$ is

$$h[n] = \frac{2f_c}{f_s}\operatorname{sinc}\!\left(\frac{2 f_c n}{f_s}\right),
\qquad n = -\frac{N-1}{2}, \dots, \frac{N-1}{2}$$

Multiply by a Hamming window to tame the ringing, then normalise so $\sum h[n] = 1$ (unit DC gain). Use $f_c = 45$ Hz, 101 taps.

**(c)** Filter with `np.convolve(x, h, mode="same")`, **then** decimate. Compare the 10 Hz amplitude to the true value of 1.

**(d)** Plot $|H(f)|$ in dB and mark $f_c$ and $f_{s2}/2$.

**(e)** Would filtering *after* decimation have helped at all?

In [ ]:
# Your turn
def lowpass_fir(fc, fs, numtaps=101):
    """Windowed-sinc lowpass built from the ideal brick-wall response."""
    # TODO
    raise NotImplementedError

### Solution 11

In [ ]:
fs_high, fs_low = 1000, 100
M = fs_high // fs_low              # decimation factor
duration = 1.0

t_high = sample_grid(duration, fs_high)
x_high = cosine(10, t_high) + 0.8 * cosine(90, t_high)    # 90 Hz will fold onto 10 Hz at fs = 100


def lowpass_fir(fc, fs, numtaps=101):
    """Windowed-sinc lowpass built from the ideal brick-wall response."""
    n = np.arange(numtaps) - (numtaps - 1) / 2            # tap indices centred on zero
    h_ideal = (2 * fc / fs) * np.sinc(2 * fc * n / fs)    # ideal lowpass impulse response
    h = h_ideal * np.hamming(numtaps)                     # taper the ends to kill ringing
    return h / h.sum()                                    # unit gain at DC


fc = 45                                      # just under the new Nyquist, fs_low/2 = 50 Hz
h = lowpass_fir(fc, fs_high, numtaps=101)
x_filtered = np.convolve(x_high, h, mode="same")

trim = 100                                   # drop the convolution's edge transients
naive = x_high[trim:-trim][::M]              # decimate only
proper = x_filtered[trim:-trim][::M]         # filter, then decimate

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
cases = [("no anti-alias filter", naive), ("with anti-alias filter", proper)]
for ax, (name, sig) in zip(axes, cases):
    _, f, mag = stem_spectrum(sig, fs_low, ax=ax)
    amp_at_10 = mag[np.argmin(np.abs(f - 10))]
    ax.set_title(f"{name}\namplitude at 10 Hz = {amp_at_10:.3f}")

fig.suptitle(f"Decimating {fs_high} Hz → {fs_low} Hz; the true 10 Hz amplitude is 1.000",
             fontsize=11)
fig.tight_layout()
plt.show()

# (d) the filter's frequency response, zero-padded to 4096 points for a smooth curve
n_fft = 4096
H = np.abs(np.fft.rfft(h, n_fft))
f_H = np.fft.rfftfreq(n_fft, d=1 / fs_high)
H_dB = 20 * np.log10(H + 1e-12)              # tiny offset avoids log(0)

plt.figure()
plt.plot(f_H, H_dB)
plt.axvline(fc, color="C2", ls="--", label=f"$f_c$ = {fc} Hz")
plt.axvline(fs_low / 2, color="C3", ls=":", label=f"$f_{{s2}}/2$ = {fs_low // 2} Hz")
plt.xlim(0, 200)
plt.ylim(-80, 5)
plt.xlabel("f (Hz)")
plt.ylabel("|H(f)| (dB)")
plt.title("Hand-built 101-tap windowed-sinc lowpass")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Discussion 11**

**(a)** On the decimated grid $t_m = m/100$,

$$\cos\!\left(2\pi \cdot 90 \cdot \frac{m}{100}\right) = \cos(2\pi m \cdot 0.9)
= \cos(2\pi m \cdot (-0.1)) = \cos\!\left(2\pi \cdot 10 \cdot \frac{m}{100}\right)$$

so the 90 Hz tone lands exactly on 10 Hz and, being in phase here, **adds**: the measured amplitude is about $1 + 0.8 = 1.8$ instead of 1. The measurement is 80% wrong and nothing in the decimated data reveals it.

**(c)** After filtering, the 90 Hz tone is attenuated by roughly 60 dB before decimation and the 10 Hz peak returns to about 1.0.

**(e)** Filtering afterwards is useless. Once the 90 Hz energy has folded onto 10 Hz, both occupy the *same* frequency, and no filter can separate two things at one frequency. The filter must run at the high rate, before the rate is reduced — which is exactly why every real ADC has an analog anti-aliasing filter in front of it.

---
## Problem 12 — Sampling plus quantisation: a one-line ADC

An ADC does two things: discretise time, and discretise amplitude.

$x(t) = \sin(2\pi \cdot 50 t)$, full scale $x_{\max} = 1$, sampled at $f_s = 5000$ Hz for 0.1 s.

**(a)** Write `quantize(x, bits, xmax=1.0)` using a mid-rise quantiser:

$$\Delta = \frac{2x_{\max}}{2^B}, \qquad q = \Delta\left\lfloor \frac{x}{\Delta}\right\rfloor + \frac{\Delta}{2}$$

then clip to the full-scale range.

**(b)** For $B = 3$, plot $x[n]$, $q[n]$ and the error $e = x - q$. Confirm $|e| \le \Delta/2$ and note that $e$ looks like a sawtooth, not like noise.

**(c)** Compute the SNR in dB as $10\log_{10}\!\left(\sum x^2 / \sum e^2\right)$.

**(d)** Sweep $B = 2 \dots 16$, plot measured SNR against the textbook line $6.02B + 1.76$ dB.

**(e)** Derive that line yourself, assuming $e$ is uniform on $[-\Delta/2, \Delta/2]$ so that $\mathbb{E}[e^2] = \Delta^2/12$.

In [ ]:
# Your turn
def quantize(x, bits, xmax=1.0):
    # TODO
    raise NotImplementedError

### Solution 12

In [ ]:
def quantize(x, bits, xmax=1.0):
    """Mid-rise uniform quantiser with 2**bits levels on [-xmax, xmax]."""
    delta = 2 * xmax / 2 ** bits                         # step size
    q = delta * np.floor(x / delta) + delta / 2          # snap to the middle of each bin
    lowest, highest = -xmax + delta / 2, xmax - delta / 2
    return np.clip(q, lowest, highest)                   # no level beyond full scale


def snr_dB(x, e):
    """Signal power over error power, in decibels."""
    return 10 * np.log10(np.sum(x ** 2) / np.sum(e ** 2))


fs = 5000
tn = sample_grid(0.1, fs)
x = np.sin(2 * np.pi * 50 * tn)

# (a), (b): a 3-bit quantiser and its error
bits = 3
delta = 2 / 2 ** bits
q = quantize(x, bits)
e = x - q

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
axes[0].plot(tn, x, color="0.65", label="x[n]")
axes[0].step(tn, q, where="post", label=f"{bits}-bit quantised")
axes[0].legend(fontsize=8)
axes[0].set_title("sampling discretises time, quantisation discretises amplitude")
axes[1].plot(tn, e, "C3", lw=0.8)
axes[1].axhline(delta / 2, color="0.5", ls="--")
axes[1].axhline(-delta / 2, color="0.5", ls="--")
axes[1].set_xlabel("t (s)")
axes[1].set_title(f"error: max |e| = {np.abs(e).max():.4f},  Δ/2 = {delta / 2:.4f}")
fig.tight_layout()
plt.show()

print(f"SNR at {bits} bits = {snr_dB(x, e):.2f} dB   (theory {6.02 * bits + 1.76:.2f} dB)")

# (c): SNR against word length
bit_list = np.arange(2, 17)
snrs = [snr_dB(x, x - quantize(x, b)) for b in bit_list]

plt.figure(figsize=(9, 4))
plt.plot(bit_list, snrs, "o-", label="measured")
plt.plot(bit_list, 6.02 * bit_list + 1.76, "--", label="$6.02B + 1.76$ dB")
plt.xlabel("bits B")
plt.ylabel("SNR (dB)")
plt.title("Quantisation SNR: about 6 dB per bit")
plt.legend()
plt.tight_layout()
plt.show()

**Discussion 12**

**(e) Derivation.** Signal power for a full-scale sine of amplitude $A = x_{\max}$:

$$P_x = \frac{A^2}{2}$$

Error power under the uniform assumption:

$$P_e = \frac{\Delta^2}{12}, \qquad \Delta = \frac{2A}{2^B}
\quad\Longrightarrow\quad P_e = \frac{4A^2}{12 \cdot 2^{2B}} = \frac{A^2}{3 \cdot 2^{2B}}$$

Therefore

$$\text{SNR} = \frac{P_x}{P_e} = \frac{A^2/2}{A^2/(3 \cdot 2^{2B})} = \frac{3}{2} 2^{2B}$$

$$\text{SNR(dB)} = 10\log_{10}(1.5) + 2B \cdot 10\log_{10}2 = 1.76 + 6.02 B$$

**Where it fits and where it does not.** The measured curve tracks the line closely for $B \gtrsim 5$. At $B = 2$ or $3$ the error is a visible sawtooth strongly correlated with the signal, so the "uniform independent noise" assumption behind the formula breaks down and the measurement sits a little off. Each extra bit halves $\Delta$, quartering the error power — hence the 6 dB per bit.

---
## Problem 13 — The wagon-wheel effect

A wheel spins at $f_0 = 8$ revolutions per second. A camera "samples" it at $f_s$ frames per second. Model the wheel as the phasor $e^{j2\pi f_0 t}$ and read off its angle each frame.

For $f_s = 10, 8, 7$ fps:

**(a)** Compute the sampled angle $\theta[n] = 2\pi f_0 n / f_s$ wrapped to $(-\pi, \pi]$.

**(b)** Unwrap with `np.unwrap`, fit a straight line, and divide the slope by $2\pi$ to get the apparent rotation rate in rev/s.

**(c)** Report the rate and its **sign** for each. Which case runs backwards? Which freezes?

**(d)** Connect each answer to the signed version of the folding formula,

$$f_{\text{apparent}} = f_0 - f_s \cdot \text{round}(f_0/f_s)$$

Why does `apparent_freq` from Problem 3 lose this sign?

In [ ]:
# Your turn
# theta = 2*np.pi*f0*n/fs
# theta_wrapped = (theta + np.pi) % (2*np.pi) - np.pi

### Solution 13

In [ ]:
f0 = 8.0                     # wheel speed, revolutions per second
frame_rates = (10, 8, 7)     # camera frame rates, frames per second
n_frames = 40

plt.figure(figsize=(9, 4))
print(f"wheel spinning at {f0:.0f} rev/s\n")

for fs in frame_rates:
    n = np.arange(n_frames)
    theta_true = 2 * np.pi * f0 * n / fs                        # angle at each frame

    # the camera only sees the angle modulo one turn
    theta_seen = (theta_true + np.pi) % (2 * np.pi) - np.pi     # wrapped into (-pi, pi]

    # the eye assumes the smallest step between frames
    theta_unwrapped = np.unwrap(theta_seen)

    # apparent speed = slope of the unwrapped angle
    rad_per_frame = np.polyfit(n, theta_unwrapped, 1)[0]
    apparent_rate = rad_per_frame * fs / (2 * np.pi)            # rev/s; the sign is the direction

    # closed form: distance to the nearest multiple of the frame rate
    predicted_rate = f0 - fs * round(f0 / fs)

    print(f"   fs = {fs:2d} fps  ->  apparent rate = {apparent_rate:+.2f} rev/s"
          f"   (formula: {predicted_rate:+.2f})")
    plt.plot(n / fs, theta_unwrapped / (2 * np.pi), ".-", label=f"fs = {fs} fps")

plt.xlabel("t (s)")
plt.ylabel("unwrapped angle (revolutions)")
plt.title("Apparent rotation = slope of the unwrapped sampled angle")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Discussion 13**

| $f_s$ | apparent rate | what you see |
|---|---|---|
| 10 fps | $8 - 10 = -2$ rev/s | wheel spins **backwards** |
| 8 fps | $8 - 8 = 0$ | wheel appears **frozen** |
| 7 fps | $8 - 7 = +1$ rev/s | forward, but far too slow |

The signed rule $f_0 - f_s \cdot \text{round}(f_0/f_s)$ predicts all three exactly.

`apparent_freq` throws the sign away because it was built for cosines, and a cosine is even — it cannot distinguish $+f$ from $-f$ (Problem 4(c) is the same fact). A rotating phasor is not even, so here the sign survives and becomes something physical: the direction of rotation.

This is the stroboscopic effect in films, and the reason wheels appear to roll backwards on video or under fluorescent lighting.

---
## Problem 14 — Bandpass sampling: beating $f_s > 2f_{\max}$

$$x(t) = \cos(2\pi \cdot 85 t) + \cos(2\pi \cdot 95 t)$$

All the energy sits in the band 80–100 Hz, so $f_{\max} = 100$ Hz but the **bandwidth** is only $B = 20$ Hz.

Plain Nyquist demands $f_s > 200$ Hz. Bandpass sampling says we can do far better, provided the aliased copies of the *band* do not overlap each other.

**(a)** Predict with `apparent_freq` where 85 Hz and 95 Hz land for $f_s = 50$ Hz and $f_s = 45$ Hz.

**(b)** Sample at each rate and plot the spectra to confirm.

**(c)** Explain which rate is usable, in terms of the band and its images at $\pm k f_s$.

**(d)** General rule: the whole band must fall inside a single Nyquist zone,

$$k\frac{f_s}{2} \le f_L \quad\text{and}\quad f_H \le (k+1)\frac{f_s}{2}
\qquad \text{for some integer } k$$

Scan $f_s$ from 40 to 120 Hz and list every rate that satisfies it.

In [ ]:
# Your turn

### Solution 14

In [ ]:
f_a, f_b = 85, 95                  # the two tones
band_lo, band_hi = 80.0, 100.0     # the band they live in
rates = (50, 45)

print("predicted landing spots")
for fs in rates:
    print(f"   fs = {fs} Hz : {f_a} Hz -> {float(apparent_freq(f_a, fs)):.1f} Hz, "
          f"{f_b} Hz -> {float(apparent_freq(f_b, fs)):.1f} Hz")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, fs in zip(axes, rates):
    tn = sample_grid(4.0, fs)
    xn = cosine(f_a, tn) + cosine(f_b, tn)
    stem_spectrum(xn, fs, ax=ax, title=f"fs = {fs} Hz")

fig.suptitle(f"Band {band_lo:.0f}–{band_hi:.0f} Hz sampled far below "
             f"$2f_{{max}}$ = {2 * band_hi:.0f} Hz", fontsize=11)
fig.tight_layout()
plt.show()


# (d) which rates keep the whole band inside one Nyquist zone?
def band_fits_one_zone(fs, band_lo, band_hi):
    """True if no multiple of fs/2 falls inside (band_lo, band_hi)."""
    zone_width = fs / 2
    k = int(np.floor(band_lo / zone_width))     # the zone that contains band_lo
    zone_top = (k + 1) * zone_width
    return band_hi <= zone_top


candidates = np.arange(40, 120.01, 0.5)
valid = [fs for fs in candidates if band_fits_one_zone(fs, band_lo, band_hi)]

print("\nusable rates between 40 and 120 Hz:")
print("  ", ", ".join(f"{v:g}" for v in valid))

**Discussion 14**

**(a), (b)** At $f_s = 50$: $85 \to 15$ Hz and $95 \to 5$ Hz. Two distinct peaks, the 10 Hz spacing preserved but the **order flipped** (spectral inversion), because the band landed in an odd-numbered Nyquist zone. At $f_s = 45$: both $85$ and $95$ land on 5 Hz. They collide, and the signal is unrecoverable.

**(c)** Chop the frequency axis into Nyquist zones of width $f_s/2$.

- $f_s = 50$: zones $[0,25], [25,50], [50,75], [75,100]$. The band $[80,100]$ sits entirely inside zone $k = 3$, so it folds down into $[0,25]$ in one piece and nothing overlaps.
- $f_s = 45$: zones $\dots, [67.5, 90], [90, 112.5]$. The band straddles the boundary at 90 Hz, so its lower and upper halves fold on top of each other.

**(d)** The scan returns $f_s = 40$ Hz and then contiguous ranges around 50–53, 67–80, 100–120 Hz. Notice that the condition automatically forces $f_s \ge 2B = 40$ Hz — the true lower limit, and five times smaller than what plain Nyquist suggested. This is how radio receivers digitise an RF band directly, without first mixing it down to baseband.

---
## Problem 15 (bonus) — Moiré: aliasing you can see

Sampling in two dimensions behaves exactly the same way. Build a **zone plate**:

$$I(x,y) = \cos\!\left(60\,(x^2 + y^2)\right), \qquad x, y \in [-1, 1] \text{ on a } 512\times512 \text{ grid}$$

Its local spatial frequency grows with radius, so it is a 2D chirp sweeping through every frequency.

**(a)** Display the full-resolution image.

**(b)** Downsample by 8 with plain slicing `I[::8, ::8]`. The rings in the outer region are aliases — they exist nowhere in the original.

**(c)** Blur first with an $8 \times 8$ box average (your anti-aliasing filter), then slice by 8. Compare.

**(d)** Relate (b) and (c) to Problem 11. What plays the role of $f_s/2$ here?

*Tip for (c):* `I.reshape(N//M, M, N//M, M).mean(axis=(1, 3))` is exactly "box-average then decimate" in one line.

In [ ]:
# Your turn

### Solution 15

In [ ]:
N = 512
M = 8                                        # decimation factor

# zone plate: a 2D chirp whose local frequency grows with radius
xx = np.linspace(-1, 1, N)
X, Y = np.meshgrid(xx, xx)
I = np.cos(60 * (X ** 2 + Y ** 2))

# (b) decimate only: keep every M-th row and column
I_naive = I[::M, ::M]

# (c) box-average each M x M block, then keep one value per block
I_blur = I.reshape(N // M, M, N // M, M).mean(axis=(1, 3))

panels = [("original 512×512", I),
          (f"decimated {M}× (no filter)", I_naive),
          ("box-filtered, then decimated", I_blur)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4.4))
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.tight_layout()
plt.show()

**Discussion 15**

The zone plate's local spatial frequency grows with radius, so beyond some radius it exceeds what the coarse grid can represent. Plain decimation folds those high frequencies back down and paints extra ring patterns that are not in the original — this is Problem 11(a) in two dimensions.

Box-averaging first is a crude lowpass; it removes the too-fine detail *before* decimating, so the outer region goes smoothly grey instead of sprouting fake rings — Problem 11(c) in two dimensions.

The role of $f_s/2$ is played by the Nyquist spatial frequency of the coarse grid: $0.5$ cycles per new pixel, i.e. $\frac{1}{2M}$ cycles per original pixel.

Camera sensors carry an optical low-pass ("anti-alias") filter for exactly this reason, and it is why fine fabric patterns shimmer on video.

---

## What to take away

- **Aliasing is irreversible.** It happens at the instant of sampling, so the fix is always a filter *before* the sampler (Problems 3, 11).
- **$f_s > 2f_{\max}$ is strict**, and the boundary case genuinely fails (Problems 2, 9).
- **Two views of the same theorem:** time domain — sinc interpolation recovers $x(t)$ exactly (Problem 8); frequency domain — the spectral copies must not overlap (Problem 7).
- **$f_{\max}$ is the wrong quantity when the signal is bandpass**; bandwidth is (Problem 14).
- **Real converters add two more effects** on top of the ideal theory: ZOH aperture droop (Problem 10) and quantisation noise at about 6 dB per bit (Problem 12).